# 0623 Sequencing Data — number provenance

Computes every figure quoted in `data_overview.md` directly from the delivered files.
Samples are referred to by **site name** (the trailing token of each result folder):

| Site | Result folder |
|---|---|
| `9-6-2` | `9-6-2/rerun_QB9Y48` |
| `9-2-1` | `9-2-1` |
| `10-10-2` | `10-10-2` |


In [1]:
import re, glob
from pathlib import Path
import pandas as pd

ROOT = Path("../../data")
GEN  = ROOT / "bacteria_genome"
MG   = ROOT / "metagenomes"

# site name -> result folder
SAMPLES = {
    "9-6-2":   GEN / "9-6-2/rerun_QB9Y48",
    "9-2-1":   GEN / "9-2-1",
    "10-10-2": GEN / "10-10-2",
}

def first(pattern):
    g = glob.glob(str(pattern))
    assert g, f"no file matches {pattern}"
    return g[0]


## Parsers
Each delivered file is plain text — these helpers pull the relevant fields.

In [2]:
def parse_bakta_txt(path):
    """Genome + annotation summary (annotation/*_reference.txt)."""
    d = {}
    for line in Path(path).read_text().splitlines():
        for key in ["Length", "GC", "coding density", "CDSs", "tRNAs", "rRNAs",
                    "ncRNAs", "hypotheticals", "pseudogenes", "CRISPR arrays"]:
            if line.strip().startswith(key + ":"):
                d[key] = line.split(":", 1)[1].strip()
    return d

def parse_kv_tsv(path):
    """Two-column key/value TSV (e.g. *_ont-stats.tsv)."""
    out = {}
    for line in Path(path).read_text().splitlines():
        p = line.split("\t")
        if len(p) >= 2:
            out[p[0]] = p[1]
    return out

def parse_header_tsv(path):
    """Header + single data row TSV (e.g. *_checkm-results.tsv)."""
    rows = Path(path).read_text().splitlines()
    return dict(zip(rows[0].split("\t"), rows[1].split("\t")))

def parse_sourmash(path):
    rows = Path(path).read_text().splitlines()
    r = dict(zip(rows[0].split(","), rows[1].split(",")))
    species = r["lineage"].split(";s__")[-1]
    return species, float(r["query_ani_at_rank"]), float(r["fraction"])


## Bacterial genomes

In [3]:
rows = []
for site, folder in SAMPLES.items():
    txt  = parse_bakta_txt(first(folder / "annotation/*_reference.txt"))
    ont  = parse_kv_tsv(first(folder / "*_ont-stats.tsv"))
    chk  = parse_header_tsv(first(folder / "*_checkm-results.tsv"))
    species, ani, frac = parse_sourmash(first(folder / "*_sourmash-genome-species.csv"))
    rows.append({
        "site": site,
        "species": species,
        "ANI": round(ani, 4),
        "genome_bp": int(txt["Length"]),
        "GC%": float(txt["GC"]),
        "coding%": float(txt["coding density"]),
        "CDS": int(txt["CDSs"]),
        "tRNA": int(txt["tRNAs"]),
        "rRNA": int(txt["rRNAs"]),
        "ncRNA": int(txt["ncRNAs"]),
        "hypo": int(txt["hypotheticals"]),
        "pseudo": int(txt["pseudogenes"]),
        "CRISPR": int(txt["CRISPR arrays"]),
        "checkM_compl": float(chk["Completeness"]),
        "checkM_contam": float(chk["Contamination"]),
    })
genome_df = pd.DataFrame(rows).set_index("site")
genome_df

,species,ANI,genome_bp,GC%,coding%,CDS,tRNA,rRNA,ncRNA,hypo,pseudo,CRISPR,checkM_compl,checkM_contam
site,,,,,,,,,,,,,,
9-6-2,Pseudomonas_E sp013179515,0.9953,6178775,62.4,89.6,5448,74,22,42,232,11,0,99.74,1.94
9-2-1,Pseudomonas_E sp013179515,0.9953,6178779,62.4,89.6,5449,74,22,42,233,11,0,99.74,1.94
10-10-2,Pseudomonas_E sp013179515,0.9975,5933216,62.6,89.7,5267,73,22,50,177,10,0,99.74,0.87


### Read / assembly QC

In [4]:
qc = []
for site, folder in SAMPLES.items():
    ont = parse_kv_tsv(first(folder / "*_ont-stats.tsv"))
    qc.append({
        "site": site,
        "reads": int(ont["total_num_reads"]),
        "bases": ont["total_seq_mbp"],
        "cov_raw": float(ont["est_cov_raw"]),
        "cov_final": float(ont["est_cov"]),
        "median_read_bp": float(ont["median_read"]),
        "median_Q": float(ont["median_phred"]),
        "read_NG50": int(ont["rNG50"]),
    })
qc_df = pd.DataFrame(qc).set_index("site")
qc_df

,reads,bases,cov_raw,cov_final,median_read_bp,median_Q,read_NG50
site,,,,,,,
9-6-2,107609,435 Mb,70.4,66.9,2484.0,24.16,51165
9-2-1,354501,1935 Mb,313.1,94.0,3436.0,25.79,66805
10-10-2,128866,522 Mb,87.9,83.5,2350.0,24.24,54751


> **Note on identity:** sites `9-6-2` and `9-2-1` have near-identical genomes
> (6,178,775 vs 6,178,779 bp; identical GC; CDS 5448 vs 5449), despite different site
> labels — `9-6-2` is the `_rerun_` assembly. `10-10-2` is ~245 kb smaller and distinct.
> All three map to one GTDB species at ANI ~0.995. See open question Q1 in `data_overview.md`.

## Metagenomes
Each `.faa` is Prodigal-predicted proteins over a (meta)SPAdes assembly. We count proteins and unique source contigs straight from the FASTA headers.

In [5]:
mg_rows = []
for faa in sorted(MG.glob("*.faa")):
    n_prot, contigs, lengths = 0, set(), []
    with open(faa) as fh:
        for line in fh:
            if line.startswith(">"):
                n_prot += 1
                m = re.match(r">(NODE_\d+)_length_(\d+)_cov", line)
                if m and m.group(1) not in contigs:
                    contigs.add(m.group(1))
                    lengths.append(int(m.group(2)))
    mg_rows.append({
        "file": faa.name,
        "predicted_proteins": n_prot,
        "source_contigs": len(contigs),
        "min_contig_bp": min(lengths),
        "max_contig_bp": max(lengths),
    })
mg_df = pd.DataFrame(mg_rows).set_index("file")
mg_df

,predicted_proteins,source_contigs,min_contig_bp,max_contig_bp
file,,,,
MG08.faa,706108,608062,286,175154
MG09.faa,650535,566380,286,58349
